In [1]:
import torch
import cv2
import torch.nn as nn
from PIL import Image
from torchvision import transforms
# Packages to download -> ipykernel, torch, opencv-python, pillow, torchvision

In [2]:
class CNN(nn.Module):
  def __init__(self, num_classes):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(3, 16, kernel_size=3, padding=1),
        nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(), nn.MaxPool2d(2)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(4096,128),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(128,num_classes)
    )

  def forward(self, x):
    x = self.features(x)
    x = self.classifier(x)
    return x

In [3]:
MODEL_PATH = "./model.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 64
model = CNN(num_classes=7).to(device)

In [4]:
def inference():
  checkpoint = torch.load(MODEL_PATH,map_location=device)
  model.load_state_dict(checkpoint["model_state"])
  model.to(device)
  model.eval()
  class_names = checkpoint["class_names"]

  print("Webcam is starting... press q to quit")
  webcam = cv2.VideoCapture(0)
  if not webcam.isOpened():
    print("Webcam error")
    return

  while True:
    ret, frame = webcam.read()

    if not ret:
      break

    rbg = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    resize_img = Image.fromarray(rbg).resize((IMG_SIZE, IMG_SIZE))
    tensor = transforms.functional.to_tensor(resize_img)
    tensor = transforms.functional.normalize(tensor,[0.485,0.456,0.406],[0.229,0.224,0.225])
    tensor = tensor.unsqueeze(0).to(device)

    with torch.no_grad():
      out = model(tensor)
      probs = torch.nn.functional.softmax(out, dim=1)
      top_prob, prob_idx = torch.max(probs, dim=1)
      label = class_names[prob_idx.item()]
      conf = top_prob.item()

    cv2.putText(frame, f"{label} {conf:.2f}", (10,30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,255,0), 2)
    cv2.imshow("Emotion Detector (q to quit)", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
      break

  webcam.release()
  cv2.destroyAllWindows()

In [5]:
inference()

Webcam is starting... press q to quit
